In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from src.visual_security.analyzer import YOLOAnalyzer, FoundryGPT4oAnalyzer, AzureAIVisionAnalyzer, SafetyAnalyzerPipeline

load_dotenv()

# Modifica questi path secondo il tuo ambiente
TEST_IMAGE = "data/test/images/10_JPG_jpg.rf.2e98bae6286cbb711d31db9c3939849a.jpg"
YOLO_WEIGHTS = "runs/detect/train/weights/best.onnx"

analyzers = []

# 1. YOLO locale (richiede training completato)
if Path(YOLO_WEIGHTS).exists():
    analyzers.append(YOLOAnalyzer(model_path=YOLO_WEIGHTS))
    print("✅ YOLO ONNX caricato")
else:
    print(f"⚠️  YOLO ONNX non trovato in {YOLO_WEIGHTS}, skip.")

# 2. Foundry GPT-4o (richiede AZURE_OPENAI_KEY in .env)
if os.getenv("AZURE_OPENAI_KEY"):
    analyzers.append(FoundryGPT4oAnalyzer(api_key=os.getenv("AZURE_OPENAI_KEY"), endpoint=os.getenv("AZURE_OPENAI_URL")))
    print("✅ Foundry GPT-4o configurato")
else:
    print("⚠️  AZURE_OPENAI_KEY non impostato, skip.")

# 3. Azure AI Vision (richiede AZURE_VISION_KEY in .env)
if os.getenv("AZURE_VISION_KEY"):
    analyzers.append(
        AzureAIVisionAnalyzer(api_key=os.getenv("AZURE_VISION_KEY"), endpoint=os.getenv("AZURE_VISION_ENDPOINT"))
    )
    print("✅ Azure AI Vision configurato")
else:
    print("⚠️  AZURE_VISION_KEY non impostato, skip.")

# Esegui l'analisi
pipeline = SafetyAnalyzerPipeline(analyzers)

if Path(TEST_IMAGE).exists():
    print(f"🚀 Avvio analisi su {TEST_IMAGE}...")
    results = pipeline.run(TEST_IMAGE)
    pipeline.print_report(results)
else:
    print(f"⚠️  Immagine non trovata: {TEST_IMAGE}")
    print("   Modifica TEST_IMAGE con il path corretto.")
